In [5]:
import os
from langchain.chat_models import init_chat_model


os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model=init_chat_model("google_genai:gemini-2.5-flash-lite")
response = model.invoke("What is the capital of France?")
print(response.content)


The capital of France is **Paris**.


In [7]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    # In a real implementation, you would call a weather API here.
    return f"The current weather in {location} is sunny with a temperature of 25°C."
try:
    model_with_tools = model.bind_tools([get_weather])
except NameError:
    model = init_chat_model("google_genai:gemini-2.5-flash-lite")
    model_with_tools = model.bind_tools([get_weather])

In [ ]:
response = model_with_tools.invoke("What is the current weather in New York?")
print(response.content)

for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call.name['name']} ")
    print(f": {tool_call.name['name']} ")



Tool called: get_weather 
Args: {'location': 'New York'} 


### tool execution loops

In [17]:
#step -1: Model generates tool calls
messages =[{"role": "user", "content": "What is the current weather in Krishnanagar?"}]
ai_msg= model_with_tools.invoke(messages)
messages.append(ai_msg)
#step -2: Execute tool calls and append results to messages
for tool_call in ai_msg.tool_calls:
   # Here we only have one tool, but in a real implementation, you would need to check the tool name and call the appropriate function.
   tool_result = get_weather.invoke(tool_call['args'])
messages.append(tool_result)
#step -2: Model generates final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)



That's good to hear! Is there anything else I can help you with regarding the weather in Krishnanagar or elsewhere?


In [20]:
messages

[{'role': 'user', 'content': 'What is the current weather in Krishnanagar?'},
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Krishnanagar"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e6881-6332-7983-9e58-125167dcee37-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Krishnanagar'}, 'id': '48fb6880-2d6a-4354-ab6c-f59b2ca6a92e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 18, 'total_tokens': 70, 'input_token_details': {'cache_read': 0}}),
 'The current weather in Krishnanagar is sunny with a temperature of 25°C.']